In [1]:
!pip install --upgrade pip setuptools wheel
!pip install seqeval evaluate tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 20.8 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 818.2/818.2 kB 35.6 MB/s eta 0:00:00
  Attempting uninstall: wheel
    Found existing installation: wheel 0.47.0
    Uninstalling wheel-0.47.0:
      Successfully uninstalled wheel-0.47.0
  Attempting uninstall: setuptools
    Found existing installation: setuptools 81.0.0
    Uninstalling setuptools-81.0.0:
      Successfully uninstalled setuptools-81.0.0
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16251 sha256=a8cf7fcd3dd6a8b4679a27c583a867e7d6ad9467332b80321a317df52a0c6aaf
  Stored in directory: /root/.cache/pi

In [2]:
import os
import json
import math
import random
import warnings
from pathlib import Path
from collections import Counter, defaultdict
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from tqdm.auto import tqdm

from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from transformers.utils import import_utils as transformers_import_utils



In [3]:
# Reproducibility
SEED = 23022006
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA devices:", torch.cuda.device_count())

Device: cuda
GPU: Tesla T4
CUDA devices: 2


In [13]:
# Config
MODEL_NAME = "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext"

MAX_LENGTH = 258
TRAIN_BATCH_SIZE = 32
EVAL_BATCH_SIZE = 32
EPOCHS = 10
VALID_RATIO = 0.1

LR_BERT = 2e-5
LR_HEAD = 1e-3
WEIGHT_DECAY = 1e-3
WARMUP_RATIO = 0.1
GRAD_CLIP = 1.0
DROPOUT_RATE = 0.5

OUTPUT_DIR = Path("/kaggle/working") 
BEST_MODEL_PATH = OUTPUT_DIR / "biomedbert_resnet_crf_ddi_best.pt"
RESNET_CHANNELS = 256
RESNET_BLOCKS = 3
RESNET_KERNEL_SIZE = 3
RESNET_DILATIONS = [1, 1, 1]
MODEL_CONFIG = {
    "channels": RESNET_CHANNELS,
    "num_blocks": RESNET_BLOCKS,
    "kernel_size": RESNET_KERNEL_SIZE,
    "dilations": RESNET_DILATIONS,
    "dropout": DROPOUT_RATE,
}
ENTITY_TYPES = ['drug', 'brand', 'group', 'drug_n']


In [8]:
TRAIN_DIR_DrugBank = Path("/kaggle/input/datasets/tuan1deptrai/traningset/DDICorpus/Train/DrugBank")
TRAIN_DIR_MedLine = Path("/kaggle/input/datasets/tuan1deptrai/traningset/DDICorpus/Train/MedLine")

TEST_ROOT = Path("/kaggle/input/datasets/tuan1deptrai/traningset/DDICorpus/Test")
TEST_DIR = TEST_ROOT / "Test for DrugNER task"

In [9]:
from pathlib import Path
from sklearn.model_selection import train_test_split


# 1. Chia danh sách file của từng nguồn
def split_xml_files(folder, dev_size=0.1, seed=42):
    xml_files = sorted(Path(folder).rglob("*.xml"))

    train_files, dev_files = train_test_split(
        xml_files,
        test_size=dev_size,
        random_state=seed,
        shuffle=True,
    )

    return train_files, dev_files
train_files_drug, dev_files_drug = split_xml_files(
    TRAIN_DIR_DrugBank, seed= SEED
)

train_files_med, dev_files_med = split_xml_files(
    TRAIN_DIR_MedLine, seed= SEED
)
test_files = sorted(TEST_DIR.rglob("*.xml"))

In [14]:
def read_xml_file(path):
    examples = []
    for sent in ET.parse(path).getroot().iter("sentence"):
        text = sent.attrib.get("text", "")
        if not text.strip():
            if sent.findall("entity"):
                raise ValueError(f"Annotation trong câu rỗng: {path}")
            continue
        entities = []
        for ent in sent.findall("entity"):
            kind = ent.attrib["type"]
            if kind not in ENTITY_TYPES:
                raise ValueError(f"Loại entity không hỗ trợ: {kind}")
            for span in ent.attrib["charOffset"].split(";"):
                start, last = map(int, span.split("-"))
                end = last + 1
                if not 0 <= start < end <= len(text):
                    raise ValueError(f"Offset XML không hợp lệ: {path}, {span}")
                entities.append(dict(start=start, end=end, type=kind, text=text[start:end]))
        examples.append(dict(id=sent.attrib.get("id", ""), document=str(path),
                             text=text, entities=entities))
    return examples

# 2. Đọc các câu từ danh sách file đã chia
def load_examples_from_files(xml_files):
    examples = []

    for xml_file in xml_files:
        examples.extend(read_xml_file(xml_file))

    return examples
train_examples_drug = load_examples_from_files(train_files_drug)
dev_examples_drug = load_examples_from_files(dev_files_drug)

train_examples_med = load_examples_from_files(train_files_med)
dev_examples_med = load_examples_from_files(dev_files_med)
# 3. Gộp hai nguồn
train_examples = train_examples_drug + train_examples_med
dev_examples = dev_examples_drug + dev_examples_med

# 4. Giữ nguyên tập test có sẵn
test_examples = load_examples_from_files(test_files)

In [15]:
# 5. Kiểm tra không trùng file giữa train và dev
train_files = train_files_drug + train_files_med
dev_files = dev_files_drug + dev_files_med

assert set(train_files).isdisjoint(dev_files)

for source, train_part, dev_part in [
    ("DrugBank", train_examples_drug, dev_examples_drug),
    ("MedLine", train_examples_med, dev_examples_med),
]:
    total = len(train_part) + len(dev_part)
    print(
        f"{source}: train={len(train_part)} câu | "
        f"dev={len(dev_part)} câu | "
        f"dev chiếm {len(dev_part) / total:.2%} số câu"
    )

print(f"\nTổng train: {len(train_examples)} câu")
print(f"Tổng dev:   {len(dev_examples)} câu")
print(f"Tổng test:  {len(test_examples)} câu")

DrugBank: train=4928 câu | dev=676 câu | dev chiếm 12.06% số câu
MedLine: train=1124 câu | dev=177 câu | dev chiếm 13.60% số câu

Tổng train: 6052 câu
Tổng dev:   853 câu
Tổng test:  665 câu


In [22]:
labels = ['O', 'B-brand', 'I-brand', 'B-drug', 'I-drug', 'B-drug_n', 'I-drug_n', 'B-group', 'I-group']
label2id = {label: idx for idx, label in enumerate(labels)}
id2label = {idx: label for label, idx in label2id.items()}

print("Train:", len(train_examples), "| Valid:", len(dev_examples), "| Test:", len(test_examples))
print("Labels:", labels)

Train: 6052 | Valid: 853 | Test: 665
Labels: ['O', 'B-brand', 'I-brand', 'B-drug', 'I-drug', 'B-drug_n', 'I-drug_n', 'B-group', 'I-group']


In [16]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
print(type(tokenizer))
print("Vocab size:", tokenizer.vocab_size)

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

<class 'transformers.models.bert.tokenization_bert.BertTokenizer'>
Vocab size: 30522


In [18]:
def align_labels_with_offsets(offsets, entities, label2id):
    token_labels = [label2id["O"] for _ in offsets]
    eval_mask = []

    for idx, (start, end) in enumerate(offsets):
        is_special = (start == 0 and end == 0)
        eval_mask.append(0 if is_special else 1)

    occupied = [False for _ in offsets]
    for ent in sorted(entities, key=lambda x: (x["start"], x["end"])):
        first_token = True
        ent_start, ent_end, ent_type = ent["start"], ent["end"], ent["type"]
        b_label = label2id[f"B-{ent_type}"]
        i_label = label2id[f"I-{ent_type}"]

        for i, (tok_start, tok_end) in enumerate(offsets):
            if tok_start == 0 and tok_end == 0:
                continue
            overlaps = tok_start < ent_end and tok_end > ent_start
            if overlaps and not occupied[i]:
                token_labels[i] = b_label if first_token else i_label
                occupied[i] = True
                first_token = False

    return token_labels, eval_mask

In [19]:
class DDINerDataset(Dataset):
    def __init__(self, examples, tokenizer, label2id, max_length=192):
        self.examples = examples
        self.tokenizer = tokenizer
        self.label2id = label2id
        self.max_length = max_length

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        ex = self.examples[idx]
        encoding = self.tokenizer(
            ex["text"],
            truncation=True,
            max_length=self.max_length,
            return_offsets_mapping=True,
            add_special_tokens=True,
        )
        offsets = encoding.pop("offset_mapping")
        if not any(end > start for start, end in offsets):
            raise ValueError(f"No labelable tokens in sentence {ex['id']!r}.")
        labels, eval_mask = align_labels_with_offsets(offsets, ex["entities"], self.label2id)

        return {
            "input_ids": torch.tensor(encoding["input_ids"], dtype=torch.long),
            "attention_mask": torch.tensor(encoding["attention_mask"], dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long),
            "eval_mask": torch.tensor(eval_mask, dtype=torch.long),
        }

In [20]:
def collate_batch(batch):
    max_len = max(item["input_ids"].size(0) for item in batch)
    pad_id = tokenizer.pad_token_id
    o_id = label2id["O"]

    out = defaultdict(list)
    for item in batch:
        length = item["input_ids"].size(0)
        pad_len = max_len - length
        out["input_ids"].append(torch.cat([item["input_ids"], torch.full((pad_len,), pad_id, dtype=torch.long)]))
        out["attention_mask"].append(torch.cat([item["attention_mask"], torch.zeros(pad_len, dtype=torch.long)]))
        out["labels"].append(torch.cat([item["labels"], torch.full((pad_len,), o_id, dtype=torch.long)]))
        out["eval_mask"].append(torch.cat([item["eval_mask"], torch.zeros(pad_len, dtype=torch.long)]))

    return {key: torch.stack(value) for key, value in out.items()}


In [23]:
train_ds = DDINerDataset(train_examples, tokenizer, label2id, MAX_LENGTH)
valid_ds = DDINerDataset(dev_examples, tokenizer, label2id, MAX_LENGTH)
test_ds = DDINerDataset(test_examples, tokenizer, label2id, MAX_LENGTH) 

train_loader = DataLoader(train_ds, batch_size=TRAIN_BATCH_SIZE, shuffle=True, collate_fn=collate_batch, num_workers=2)
dev_loader = DataLoader(valid_ds, batch_size=EVAL_BATCH_SIZE, shuffle=False, collate_fn=collate_batch, num_workers=2)
test_loader = DataLoader(test_ds, batch_size=EVAL_BATCH_SIZE, shuffle=False, collate_fn=collate_batch, num_workers=2) if test_ds else None

batch = next(iter(train_loader))
print({k: tuple(v.shape) for k, v in batch.items()})
print("Decoded sample:", tokenizer.convert_ids_to_tokens(batch["input_ids"][0][:30]))
print("Label sample:", [id2label[i.item()] for i in batch["labels"][0][:30]])

{'input_ids': (32, 58), 'attention_mask': (32, 58), 'labels': (32, 58), 'eval_mask': (32, 58)}
Decoded sample: ['[CLS]', 'no', 'pharmacokinetic', 'data', 'are', 'available', '.', '[SEP]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]']
Label sample: ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']


## Kiến trúc

In [24]:
def _validate_mask(mask: torch.Tensor, shape: tuple[int, int]) -> torch.Tensor:
    if mask.shape != shape or len(shape) != 2 or min(shape) == 0:
        raise ValueError("Expected a nonempty [batch, length] mask.")
    if not ((mask == 0) | (mask == 1)).all():
        raise ValueError("Masks must contain only 0/1 values.")
    mask = mask.bool()
    if not mask[:, 0].all() or (mask[:, 1:] & ~mask[:, :-1]).any():
        raise ValueError("Each sequence must be nonempty and right padded.")
    return mask

In [25]:
"""Length-preserving residual Conv1D + linear-chain CRF for flat BIO NER.

ResNetCRF accepts word/subword features; BertResNetCRF adds a Hugging Face
encoder. ``tag_mask`` selects positions to label (exclude special tokens or
select first subwords for word-level tagging). Decoding returns -100 at
unselected positions and omits right padding, preserving input alignment.
"""
from collections.abc import Sequence
import torch
from torch import nn
from torch.nn.utils.rnn import pad_sequence

class LinearChainCRF(nn.Module):
    def __init__(self, num_tags: int, labels: Sequence[str] | None = None):
        super().__init__()
        if num_tags < 1:
            raise ValueError("num_tags must be positive.")
        self.num_tags = num_tags
        self.start_transitions = nn.Parameter(torch.empty(num_tags))
        self.end_transitions = nn.Parameter(torch.empty(num_tags))
        self.transitions = nn.Parameter(torch.empty(num_tags, num_tags))
        start_allowed = torch.ones(num_tags, dtype=torch.bool)
        allowed = torch.ones(num_tags, num_tags, dtype=torch.bool)
        if labels is not None:
            if len(labels) != num_tags or len(set(labels)) != num_tags or "O" not in labels:
                raise ValueError("Provide unique BIO labels matching num_tags, including O.")
            for j, label in enumerate(labels):
                if label != "O" and (label[:2] not in ("B-", "I-") or not label[2:]):
                    raise ValueError(f"Not a BIO label: {label!r}")
                if label.startswith("I-"):
                    if f"B-{label[2:]}" not in labels:
                        raise ValueError(f"Missing B tag for {label!r}.")
                    start_allowed[j] = False
                    for i, previous in enumerate(labels):
                        allowed[i, j] = previous in (f"B-{label[2:]}", label)
        self.register_buffer("start_allowed", start_allowed)
        self.register_buffer("transition_allowed", allowed)
        for parameter in self.parameters():
            nn.init.uniform_(parameter, -0.1, 0.1)

    def _prepare(self, emissions, mask):
        if emissions.ndim != 3 or emissions.size(-1) != self.num_tags:
            raise ValueError("emissions must have shape [batch, length, num_tags].")
        mask = _validate_mask(mask, emissions.shape[:2])
        # Dynamic programming stays in float32 even inside mixed precision.
        emissions = emissions.float().masked_fill(~mask.unsqueeze(-1), 0.0)
        start = self.start_transitions.float().masked_fill(~self.start_allowed, -torch.inf)
        transitions = self.transitions.float().masked_fill(~self.transition_allowed, -torch.inf)
        return emissions, mask, start, transitions

    def forward(self, emissions, tags, mask):
        emissions, mask, start, transitions = self._prepare(emissions, mask)
        if tags.shape != mask.shape or tags.dtype != torch.long:
            raise ValueError("tags must be int64 with shape [batch, length].")
        if ((tags[mask] < 0) | (tags[mask] >= self.num_tags)).any():
            raise ValueError("Valid positions must contain tag IDs, not ignore indices.")
        tags = tags.masked_fill(~mask, 0)
        if not self.start_allowed[tags[:, 0]].all():
            raise ValueError("Gold BIO sequences cannot start with I tags.")
        if ((~self.transition_allowed[tags[:, :-1], tags[:, 1:]]) & mask[:, 1:]).any():
            raise ValueError("Gold sequence contains an invalid BIO transition.")

        batch = torch.arange(emissions.size(0), device=emissions.device)
        gold = start[tags[:, 0]] + emissions[batch, 0, tags[:, 0]]
        score = start + emissions[:, 0]
        for t in range(1, emissions.size(1)):
            step = transitions[tags[:, t - 1], tags[:, t]] + emissions[batch, t, tags[:, t]]
            gold = gold + torch.where(mask[:, t], step, 0.0)
            candidates = score.unsqueeze(2) + transitions + emissions[:, t].unsqueeze(1)
            next_score = torch.logsumexp(candidates, dim=1)
            score = torch.where(mask[:, t, None], next_score, score)
        last = mask.long().sum(1) - 1
        gold = gold + self.end_transitions.float()[tags[batch, last]]
        partition = torch.logsumexp(score + self.end_transitions.float(), dim=1)
        return (partition - gold).mean()

    @torch.no_grad()
    def decode(self, emissions, mask):
        emissions, mask, start, transitions = self._prepare(emissions, mask)
        score = start + emissions[:, 0]
        history = []
        for t in range(1, emissions.size(1)):
            candidates = score.unsqueeze(2) + transitions + emissions[:, t].unsqueeze(1)
            best_score, previous = candidates.max(dim=1)
            score = torch.where(mask[:, t, None], best_score, score)
            history.append(previous)
        last_tags = (score + self.end_transitions.float()).argmax(dim=1)
        paths = []
        for b, length in enumerate(mask.sum(1).tolist()):
            tag = last_tags[b].item()
            path = [tag]
            for previous in reversed(history[:length - 1]):
                tag = previous[b, tag].item()
                path.append(tag)
            paths.append(path[::-1])
        return paths

In [26]:
class ResidualConv1DBlock(nn.Module):
    """Two same-length convolutions; LayerNorm is applied per position.

    Masking after every convolution prevents padded activations from leaking
    back into valid tokens in the next layer. LayerNorm avoids statistics
    depending on padding length or the other examples in the batch.
    """

    def __init__(self, channels: int, kernel_size: int = 3, dilation: int = 1,
                 dropout: float = 0.2):
        super().__init__()
        if kernel_size < 1 or kernel_size % 2 == 0 or dilation < 1:
            raise ValueError("Use a positive odd kernel size and positive dilation.")
        padding = dilation * (kernel_size - 1) // 2
        self.conv1 = nn.Conv1d(channels, channels, kernel_size, padding=padding, dilation=dilation)
        self.conv2 = nn.Conv1d(channels, channels, kernel_size, padding=padding, dilation=dilation)
        self.norm1 = nn.LayerNorm(channels)
        self.norm2 = nn.LayerNorm(channels)
        self.activation = nn.ReLU()
        self.dropout = nn.Dropout(dropout)

    def forward(self, features, mask):
        padding = ~mask.bool().unsqueeze(-1)
        residual = features.masked_fill(padding, 0.0)
        hidden = self.conv1(residual.transpose(1, 2)).transpose(1, 2)
        hidden = self.dropout(self.activation(self.norm1(hidden))).masked_fill(padding, 0.0)
        hidden = self.conv2(hidden.transpose(1, 2)).transpose(1, 2)
        hidden = self.dropout(self.norm2(hidden)).masked_fill(padding, 0.0)
        return self.activation(residual + hidden).masked_fill(padding, 0.0)

In [27]:
class ResNetCRF(nn.Module):
    """Reusable NER head for [batch, length, input_size] contextual features."""

    def __init__(self, input_size: int, num_labels: int, channels: int = 256,
                 num_blocks: int = 3, kernel_size: int = 3, dropout: float = 0.2,
                 dilations: Sequence[int] | None = None,
                 labels: Sequence[str] | None = None):
        super().__init__()
        if min(input_size, channels, num_blocks) < 1:
            raise ValueError("input_size, channels and num_blocks must be positive.")
        dilations = tuple(dilations) if dilations is not None else (1,) * num_blocks
        if len(dilations) != num_blocks:
            raise ValueError("Provide one dilation per residual block.")
        self.input_projection = nn.Linear(input_size, channels)
        self.input_norm = nn.LayerNorm(channels)
        self.dropout = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([
            ResidualConv1DBlock(channels, kernel_size, dilation, dropout)
            for dilation in dilations
        ])
        self.classifier = nn.Linear(channels, num_labels)
        self.crf = LinearChainCRF(num_labels, labels)

    def emissions(self, features, attention_mask):
        if features.ndim != 3:
            raise ValueError("features must have shape [batch, length, input_size].")
        mask = _validate_mask(attention_mask, features.shape[:2])
        # Clear padding before projection as well as after each residual layer.
        features = features.masked_fill(~mask.unsqueeze(-1), 0.0)
        hidden = self.dropout(self.input_norm(self.input_projection(features)))
        for block in self.blocks:
            hidden = block(hidden, mask)
        return self.classifier(self.dropout(hidden)).masked_fill(~mask.unsqueeze(-1), 0.0)

    @staticmethod
    def _pack(emissions, attention_mask, tag_mask=None, labels=None):
        mask = _validate_mask(attention_mask, emissions.shape[:2])
        selected = mask if tag_mask is None else tag_mask
        if selected.shape != mask.shape or not ((selected == 0) | (selected == 1)).all():
            raise ValueError("tag_mask must be a binary mask matching attention_mask.")
        selected = selected.bool()
        if (selected & ~mask).any() or not selected.any(dim=1).all():
            raise ValueError("Select at least one non-padding token in every sequence.")
        if labels is not None and (labels.shape != mask.shape or labels.dtype != torch.long):
            raise ValueError("labels must be int64 with shape [batch, length].")
        packed = pad_sequence([row[keep] for row, keep in zip(emissions, selected)], batch_first=True)
        lengths = selected.sum(1)
        packed_mask = torch.arange(packed.size(1), device=emissions.device)[None] < lengths[:, None]
        packed_labels = None
        if labels is not None:
            packed_labels = pad_sequence([row[keep] for row, keep in zip(labels, selected)],
                                         batch_first=True, padding_value=-100)
        return packed, packed_mask, packed_labels, selected

    def loss_from_emissions(self, emissions, labels, attention_mask, tag_mask=None):
        packed, mask, tags, _ = self._pack(emissions, attention_mask, tag_mask, labels)
        return self.crf(packed, tags, mask)

    @torch.no_grad()
    def decode_emissions(self, emissions, attention_mask, tag_mask=None):
        packed, mask, _, selected = self._pack(emissions, attention_mask, tag_mask)
        paths = self.crf.decode(packed, mask)
        aligned = []
        for b, path in enumerate(paths):
            row = [-100] * int(attention_mask[b].sum().item())
            for position, tag in zip(selected[b].nonzero(as_tuple=True)[0].tolist(), path):
                row[position] = tag
            aligned.append(row)
        return aligned

    def forward(self, features, attention_mask, labels=None, tag_mask=None):
        emissions = self.emissions(features, attention_mask)
        if labels is None:
            return emissions
        return self.loss_from_emissions(emissions, labels, attention_mask, tag_mask), emissions

    @torch.no_grad()
    def decode(self, features, attention_mask, tag_mask=None):
        return self.decode_emissions(self.emissions(features, attention_mask), attention_mask, tag_mask)



In [28]:
class BertResNetCRF(nn.Module):
    """Pretrained text encoder -> residual Conv1D head -> CRF.
    An encoder can be injected for offline testing. It must expose
    config.hidden_size and return an object with last_hidden_state.
    """

    def __init__(self, model_name: str | None, num_labels: int, *, encoder=None, **head_kwargs):
        super().__init__()
        if encoder is None:
            if model_name is None:
                raise ValueError("Provide model_name or encoder.")
            from transformers import AutoModel
            encoder = AutoModel.from_pretrained(model_name)
        self.bert = encoder
        self.head = ResNetCRF(self.bert.config.hidden_size, num_labels, **head_kwargs)

    def emissions(self, input_ids, attention_mask):
        _validate_mask(attention_mask, input_ids.shape)
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        return self.head.emissions(outputs.last_hidden_state, attention_mask)

    def forward(self, input_ids, attention_mask, labels=None, tag_mask=None):
        emissions = self.emissions(input_ids, attention_mask)
        if labels is None:
            return emissions
        return self.head.loss_from_emissions(emissions, labels, attention_mask, tag_mask), emissions

    @torch.no_grad()
    def decode(self, input_ids, attention_mask, tag_mask=None):
        return self.head.decode_emissions(self.emissions(input_ids, attention_mask), attention_mask, tag_mask)


In [31]:
model = BertResNetCRF(
    MODEL_NAME, num_labels=len(labels), labels=labels, **MODEL_CONFIG,
).to(device)
print("Trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))
print("ResNet/CRF parameters:", sum(p.numel() for p in model.head.parameters()))


pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

BertModel LOAD REPORT from: microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Trainable parameters: 110866284
ResNet/CRF parameters: 1384044


In [40]:
from seqeval.metrics import precision_score, recall_score, f1_score
from seqeval.scheme import IOB2
import torch
from tqdm import tqdm

def evaluate(model, dataloader, id2label, desc="Evaluating", device="cuda"):
    model.eval()
    total_loss = 0.0
    n_batches = 0
    token_correct = 0
    token_total = 0

    
    all_gold_tags = []
    all_pred_tags = []

    with torch.no_grad():
        for batch in tqdm(dataloader, desc=desc):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels_tensor = batch["labels"].to(device)
            eval_mask = batch["eval_mask"].to(device)

            loss, _ = model(input_ids, attention_mask, labels_tensor)
            paths = model.decode(input_ids, attention_mask)

            total_loss += loss.item()
            n_batches += 1

            for b, path in enumerate(paths):
                gold_tags = []
                pred_tags = []
                valid_positions = eval_mask[b].bool().cpu().tolist()
                gold_ids = labels_tensor[b].detach().cpu().tolist()

                for pos, keep in enumerate(valid_positions[:len(path)]):
                    if not keep:
                        continue
                    gold = id2label[gold_ids[pos]]
                    pred = id2label[path[pos]]
                    
                    gold_tags.append(gold)
                    pred_tags.append(pred)
                    
                    token_correct += int(gold == pred)
                    token_total += 1

                # Thêm chuỗi nhãn của câu hiện tại vào list tổng
                # Chỉ thêm nếu câu có chứa token hợp lệ (để tránh mảng rỗng)
                if gold_tags:
                    all_gold_tags.append(gold_tags)
                    all_pred_tags.append(pred_tags)

    # ---------------------------------------------------------
    # SỬ DỤNG SEQEVAL ĐỂ TÍNH TOÁN METRICS
    # ---------------------------------------------------------
    
    # 1. Tính Micro F1 (Đồng nhất với cách tính cũ của bạn - gom chung toàn bộ entity)
    micro_precision = precision_score(all_gold_tags, all_pred_tags, mode="strict", scheme=IOB2, average="micro")
    micro_recall = recall_score(all_gold_tags, all_pred_tags, mode="strict", scheme=IOB2, average="micro")
    micro_f1 = f1_score(all_gold_tags, all_pred_tags, mode="strict", scheme=IOB2, average="micro")

    # 2. Tính Macro F1 (Tính F1 riêng cho từng loại entity như PER, ORG, LOC... rồi lấy trung bình cộng)
    macro_f1 = f1_score(all_gold_tags, all_pred_tags, mode="strict", scheme=IOB2, average="macro")

    token_acc = token_correct / token_total if token_total else 0.0

    return {
        "loss": float(total_loss / max(n_batches, 1)),
        "token_acc": float(token_acc),
        "precision": float(micro_precision),
        "recall": float(micro_recall),
        "f1": float(micro_f1),
        "macro_f1": float(macro_f1),
    }

In [38]:
import torch
from tqdm import tqdm

def train_epoch(model, train_loader, optimizer, scheduler, scaler, device, epoch, total_epochs, grad_clip):
    """
    Thực thi quá trình huấn luyện trong 1 epoch.
    """
    model.train()
    train_loss = 0.0
    examples_seen = 0
    progress = tqdm(train_loader, desc=f"Epoch {epoch}/{total_epochs}")
    
    for batch in progress:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        gold = batch["labels"].to(device)
        tag_mask = batch["eval_mask"].to(device)
        
        optimizer.zero_grad(set_to_none=True)
        
        # Forward pass với AMP
        with torch.autocast(device_type=device.type, enabled=(device.type == "cuda")):
            loss, _ = model(input_ids, attention_mask, gold, tag_mask=tag_mask)
            
        if not torch.isfinite(loss):
            raise RuntimeError("Non-finite training loss.")
            
        # Backward pass và cập nhật trọng số an toàn
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        
        previous_scale = scaler.get_scale()
        scaler.step(optimizer)
        scaler.update()
        
        # Bỏ qua step của scheduler nếu AMP bỏ qua lần cập nhật này do tràn số (overflow)
        if scaler.get_scale() >= previous_scale:
            scheduler.step()  
            
        # Tính toán loss trung bình và cập nhật thanh tiến trình
        batch_size = input_ids.size(0)
        train_loss += loss.item() * batch_size
        examples_seen += batch_size
        progress.set_postfix(loss=f"{train_loss / examples_seen:.4f}")

    return train_loss / examples_seen

In [33]:
bert_params = list(model.bert.parameters())
head_params = list(model.head.parameters())

optimizer = AdamW(
    [
        {"params": bert_params, "lr": LR_BERT, "weight_decay": WEIGHT_DECAY},
        {"params": head_params, "lr": LR_HEAD, "weight_decay": WEIGHT_DECAY},
    ]
)

total_steps = len(train_loader) * EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)

scaler = torch.amp.GradScaler("cuda", enabled=(device.type == "cuda"))
print("Total steps:", total_steps, "| Warmup steps:", warmup_steps)

Total steps: 1900 | Warmup steps: 190


In [34]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("training_dii")

In [37]:
import wandb

WANDB_PROJECT = "ddi-ner-embedAttention_BiLSTM_CRF"
WANDB_RUN_NAME = f"BERT_ResNet1D_CRF"


def wandb_config():
    return {
        "architecture": "BiomedBERT + ResNet1D + CRF",
        "max_length": MAX_LENGTH,
        "train_batch_size": TRAIN_BATCH_SIZE,
        "eval_batch_size": EVAL_BATCH_SIZE,
        "epochs": EPOCHS,
        "valid_ratio": VALID_RATIO,
        "lr_bert": LR_BERT,
        "lr_head": LR_HEAD,
        "weight_decay": WEIGHT_DECAY,
        "warmup_ratio": WARMUP_RATIO,
        "grad_clip": GRAD_CLIP,
        "seed": SEED,
        "num_labels": len(labels),
        "labels": labels,
        "train_sentences": len(train_examples),
        "valid_sentences": len(dev_examples),
        "test_sentences": len(test_examples),
    }


def ensure_wandb_run():
    if wandb.run is None:
        run = wandb.init(
            project=WANDB_PROJECT,
            name=WANDB_RUN_NAME,
            config=wandb_config(),
        )
        wandb.watch(model, log="gradients", log_freq=100)
        return run
    return wandb.run

wandb.login(key=secret_value_0)
wandb_run = ensure_wandb_run()


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


In [42]:
import pandas as pd
import torch

best_dev_macro_f1 = -1.0
history = []

for epoch in range(1, EPOCHS + 1):
    # 1. Chạy huấn luyện cho epoch hiện tại
    train_loss = train_epoch(
        model=model,
        train_loader=train_loader,
        optimizer=optimizer,
        scheduler=scheduler,
        scaler=scaler,
        device=device,
        epoch=epoch,
        total_epochs=EPOCHS,
        grad_clip=GRAD_CLIP
    )

    # 2. Đánh giá trên tập Validation
    dev_metrics = evaluate(model, dev_loader, id2label, desc=f"Dev eval epoch {epoch}", device=device)
    
    # 3. Ghi nhận lịch sử huấn luyện
    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        **{f"dev_{k}": v for k, v in dev_metrics.items()},
    }
    history.append(row)
    print(row)
    wandb.log(
        row,
        step=epoch,
    )
    # 4. Kiểm tra và lưu checkpoint nếu mô hình tốt hơn
    if dev_metrics["macro_f1"] > best_dev_macro_f1:
        best_dev_macro_f1 = dev_metrics["macro_f1"]
        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "label2id": label2id,
                "id2label": id2label,
                "model_name": MODEL_NAME,
                "max_length": MAX_LENGTH,
                "valid_metrics": dev_metrics,
            },
            BEST_MODEL_PATH,
        )
        wandb.run.summary["best_dev_macro_f1"] = best_dev_macro_f1
        wandb.run.summary["best_epoch"] = epoch
        wandb.run.summary["best_model_path"] = str(BEST_MODEL_PATH)
        print("Saved best model to", BEST_MODEL_PATH)

# Trả về bảng tổng hợp sau khi train xong
history_df = pd.DataFrame(history)
wandb.log({"history": wandb.Table(dataframe=history_df)})
wandb.finish()
history_df

Dev eval epoch 1: 100%|██████████| 27/27 [00:08<00:00,  3.16it/s]


{'epoch': 1, 'train_loss': 1.2657816085449844, 'dev_loss': 1.2165376168710214, 'dev_token_acc': 0.9706642847068779, 'dev_precision': 0.9041406693136699, 'dev_recall': 0.9061967026719727, 'dev_f1': 0.9051675184554231, 'dev_macro_f1': 0.7810168399642083}
Saved best model to /kaggle/working/biomedbert_resnet_crf_ddi_best.pt


Dev eval epoch 2: 100%|██████████| 27/27 [00:08<00:00,  3.35it/s]


{'epoch': 2, 'train_loss': 0.785468034845114, 'dev_loss': 1.1129554587381858, 'dev_token_acc': 0.9732969771049786, 'dev_precision': 0.9110604332953249, 'dev_recall': 0.908470722001137, 'dev_f1': 0.9097637346996869, 'dev_macro_f1': 0.8089460947506402}
Saved best model to /kaggle/working/biomedbert_resnet_crf_ddi_best.pt


Dev eval epoch 3: 100%|██████████| 27/27 [00:08<00:00,  3.26it/s]


...

{'epoch': 9, 'train_loss': 0.15511091418874556, 'dev_loss': 1.7957044045130413, 'dev_token_acc': 0.9777631517088995, 'dev_precision': 0.9212778094694809, 'dev_recall': 0.9181353041500853, 'dev_f1': 0.9197038724373576, 'dev_macro_f1': 0.8502562848893503}


Dev eval epoch 10: 100%|██████████| 27/27 [00:08<00:00,  3.28it/s]


{'epoch': 10, 'train_loss': 0.14129765968959376, 'dev_loss': 1.7958393273530182, 'dev_token_acc': 0.9777631517088995, 'dev_precision': 0.9212778094694809, 'dev_recall': 0.9181353041500853, 'dev_f1': 0.9197038724373576, 'dev_macro_f1': 0.8502562848893503}


dev_f1,▁▃▆▅█▄▅█▇▇
dev_loss,▂▁▁▂▄▄▆███
dev_macro_f1,▁▄█▇▆▆▇███
dev_precision,▁▃▅██▄▅▇▆▆
dev_recall,▁▂█▁▅▂▄█▇▇
dev_token_acc,▁▃▇▅▇▅▆█▇▇
epoch,▁▂▃▃▄▅▆▆▇█
train_loss,█▅▄▃▂▂▁▁▁▁
best_dev_macro_f1,0.85474
best_epoch,3
best_model_path,/kaggle/working/biom...


,epoch,train_loss,dev_loss,dev_token_acc,dev_precision,dev_recall,dev_f1,dev_macro_f1
0,1,1.265782,1.216538,0.970664,0.904141,0.906197,0.905168,0.781017
1,2,0.785468,1.112955,0.973297,0.911060,0.908471,0.909764,0.808946
2,3,0.588185,1.144584,0.977434,0.916242,0.920409,0.918321,0.854743
3,4,0.432672,1.213863,0.975366,0.926206,0.906197,0.916092,0.839210
4,5,0.346939,1.445751,0.977152,0.927419,0.915293,0.921316,0.831019
5,6,0.281885,1.432814,0.975130,0.914237,0.909039,0.911631,0.836701
6,7,0.218833,1.619775,0.976071,0.918764,0.913019,0.915883,0.840603
7,8,0.178443,1.789682,0.978562,0.924571,0.919841,0.922200,0.854123
8,9,0.155111,1.795704,0.977763,0.921278,0.918135,0.919704,0.850256
9,10,0.141298,1.795839,0.977763,0.921278,0.918135,0.919704,0.850256


In [43]:
# Load best checkpoint before final evaluation.
checkpoint = torch.load(BEST_MODEL_PATH, map_location=device, weights_only=True)
model.load_state_dict(checkpoint["model_state_dict"])
print("Best validation metrics:", checkpoint["valid_metrics"])

Best validation metrics: {'loss': 1.1445842054155138, 'token_acc': 0.9774340651591369, 'precision': 0.9162422184493492, 'recall': 0.9204093234792495, 'f1': 0.918321043675553, 'macro_f1': 0.8547434950172106}


In [44]:
test_metrics = evaluate(model, test_loader, id2label, desc=f"Test", device=device)
print(f"Test loss: {test_metrics['loss']}")
print(f"Test token_acc: {test_metrics['token_acc']}")
print(f"Test precision: {test_metrics['precision']}")
print(f"Test micro_f1: {test_metrics['f1']}")
print(f"Test macro_f1: {test_metrics['macro_f1']}")

Test: 100%|██████████| 21/21 [00:06<00:00,  3.48it/s]


Test loss: 1.5696661812918526
Test token_acc: 0.968640514617196
Test precision: 0.7379679144385026
Test micro_f1: 0.7698744769874477
Test macro_f1: 0.7360394417581089


In [46]:
test_drug = Path("/kaggle/input/datasets/tuan1deptrai/traningset/DDICorpus/Test/Test for DrugNER task/DrugBank")
test_med = Path("/kaggle/input/datasets/tuan1deptrai/traningset/DDICorpus/Test/Test for DrugNER task/MedLine")

test_files_drug = sorted(test_drug.rglob("*.xml"))
test_files_med = sorted(test_med.rglob("*.xml"))

test_examples_drug = load_examples_from_files(test_files_drug)
test_examples_med = load_examples_from_files(test_files_med)

test_ds_drug = DDINerDataset(test_examples_drug, tokenizer, label2id, MAX_LENGTH) 
test_ds_med = DDINerDataset(test_examples_med, tokenizer, label2id, MAX_LENGTH) 

test_loader_drug = DataLoader(test_ds_drug, batch_size=EVAL_BATCH_SIZE, shuffle=False, collate_fn=collate_batch, num_workers=2) 
test_loader_med = DataLoader(test_ds_med, batch_size=EVAL_BATCH_SIZE, shuffle=False, collate_fn=collate_batch, num_workers=2) 

test_metrics_drug = evaluate(model, test_loader_drug, id2label, desc=f"Test", device=device)
print(f"Test_Drug loss: {test_metrics_drug['loss']}")
print(f"Test_Drug token_acc: {test_metrics_drug['token_acc']}")
print(f"Test_Drug precision: {test_metrics_drug['precision']}")
print(f"Test_Drug micro_f1: {test_metrics_drug['f1']}")
print(f"Test_Drug macro_f1: {test_metrics_drug['macro_f1']}")

test_metrics_med = evaluate(model, test_loader_med, id2label, desc=f"Test", device=device)
print(f"Test_Med loss: {test_metrics_med['loss']}")
print(f"Test_Med token_acc: {test_metrics_med['token_acc']}")
print(f"Test_Med precision: {test_metrics_med['precision']}")
print(f"Test_Med micro_f1: {test_metrics_med['f1']}")
print(f"Test_Med macro_f1: {test_metrics_med['macro_f1']}")

Test: 100%|██████████| 5/5 [00:01<00:00,  3.14it/s]


Test_Drug loss: 1.354009222984314
Test_Drug token_acc: 0.9701657458563536
Test_Drug precision: 0.8825396825396825
Test_Drug micro_f1: 0.8982229402261712
Test_Drug macro_f1: 0.672383399141003


Test: 100%|██████████| 17/17 [00:04<00:00,  3.54it/s]


Test_Med loss: 1.6501085547839893
Test_Med token_acc: 0.9682401566238852
Test_Med precision: 0.6327944572748267
Test_Med micro_f1: 0.6723926380368098
Test_Med macro_f1: 0.6487469643864097
